## Step 2 - Pocket dataframes, Global ID assignment & apo/holo comparison
This is the second explanatory notebook of the pipeline.

Step 1 left every replicate's `pockets/` folder full of mdpocket/ATClus output (dummy-atom PDBs, per-frame descriptor files, residue files). This notebook turns that into per-pocket dataframes, assigns Global IDs across replicates/genes/states, and compares apo vs holo pocketomes.

Two scripts, run in sequence:

1. **`pipeline/pocket_dataframes.py`** (Step 2.1) - parses every pocket, computes the interpolated volume, orthosteric/transient/largest-pocket flags and volume category, and writes two dataframes: `all_pockets` (every pocket, every frame, every alpha-sphere coordinate) and `pocket_summary` (one row per pocket, its first frame).
2. **`pipeline/global_id_and_comparison.py`** (Step 2.2) - clusters a *chosen subset* of `all_pockets` (by state / gene / PDB ID) into Global IDs via voxel-IoU overlap, then (optionally) compares apo vs holo pocketomes for that subset.

Both scripts write every big table as *both* `.csv` and `.parquet` (`pipeline/pocket_io.py`) - read whichever you prefer, `pocket_io.load_table()` picks parquet by default and falls back to csv.

In [ ]:
import os, sys
import importlib
import time
import warnings

from pipeline import pocket_io
from pipeline import pocket_dataframes
from pipeline import global_id_and_comparison as gid
import config as conf

importlib.reload(pocket_io)
importlib.reload(pocket_dataframes)
importlib.reload(gid)
importlib.reload(conf)

warnings.filterwarnings("ignore")


Define the global variables. `SAVING_LOC` is where Step 2.1's `all_pockets` / `pocket_summary` land (default `output/meta_analysis/across_genes/`); Global ID runs each get their own subdirectory under `output/meta_analysis/` so different subsets never mix numbering (see Step 2.2 below).

In [ ]:
# Globals
VERBOSE = True
MAKE_PLOTS = True
FORMATS = ('csv', 'parquet')
SAVING_LOC = conf.META_ANALYSIS_DIR   # output/meta_analysis/across_genes

BW_FILE_LOC = os.path.join(conf.REFERENCE_DATA_DIR, 'TAARs_numbered')
PDB_FILE_LOC = os.path.join(conf.HOLO_RESULTS_DIR, 'holo8ITF', '1')  # backbone outline for the largest/orthosteric-pocket plots

# Scopes Step 2.2's Global ID clustering (not Step 2.1's parse, which always covers every PDB
# ID) to a specific, meaningful research question -- here, one PDB ID per gene, to showcase
# Global ID conservation/divergence across a biologically diverse set of genes, rather than an
# unscoped run across every solved structure of the same receptor (redundant near-duplicates,
# not a deliberate comparison). Set to None to cluster every PDB ID together instead -- that
# needs a much larger-memory machine (100+GB; has OOM-killed a 230GB+ node).
PDB_IDS = conf.REPRESENTATIVE_PDB_IDS


## Step 2.1 - Building the pocket dataframes
`pocket_dataframes.pocket_dirs_for()` walks `output/{apo,holo}_structures/<state><PDBID>/<rep>/pockets/` (Step 1's output) and returns every replicate that has one -- every PDB ID, always: `all_pockets`/`pocket_summary` are meant to be the complete dataset regardless of which subset any given Global ID run (Step 2.2, below) is scoped to. `build_pocket_dataframes()` then:

- parses the dummy-atom/descriptor/residue files (`PocketFileParser`)
- interpolates `pock_volume` across short closures -> `interpolated_pock_volume`
- classifies orthosteric/binding-site pockets (`orthosteric_filter_ligand_based.classify_binding_site`)
- flags the largest pocket per (state, PDB ID, replicate) experiment, and `is_largest_and_orthosteric`
- classifies transient/stable pockets (`consecutive_zeros_transiency.classify_transient`)
- derives `volume_category` from the per-pocket trajectory-median volume (open frames only)

 per pocket and writes `all_pockets` and `pocket_summary` (+ `orthosteric_perframe_volumes.csv`).

In [ ]:
pocket_dirs = pocket_dataframes.pocket_dirs_for()  # every PDB ID -- see markdown above
print(f'Found {len(pocket_dirs)} pocket directories')
pocket_dirs


In [ ]:
result = pocket_dataframes.build_pocket_dataframes(
    pocket_dirs,
    saving_loc=SAVING_LOC,
    formats=FORMATS,
    make_plots=MAKE_PLOTS,
    bw_file_loc=BW_FILE_LOC,
    pdb_file_loc=PDB_FILE_LOC,
    verbose=VERBOSE,
)
all_pockets, pocket_summary = result['all_pockets'], result['pocket_summary']
pocket_summary.head()


## Step 2.2 - Global ID assignment

A pocket's Global ID is only meaningful **within the run that produced it**. Two pockets can only share a Global ID if they were voxel-clustered together, so Global IDs from two different runs are never directly comparable.

`global_id_and_comparison.run_global_id_states()` runs one or more clustering runs in a single call - pass `states` as a list made up of any of `'apo'`, `'holo'`, `'both'` (any combination, any order):

- **`'both'`** (the default when `states=None`) - apo and holo clustered *together*, sharing one Global ID numbering, written to `global_ID_combined/`. This also runs the apo/holo pocketome comparison automatically (needs both states in the same run).
- **`'apo'`** / **`'holo'`** - cluster one state alone, written to its own `global_ID_apo/` / `global_ID_holo/`.
- Request **both `'apo'` and `'holo'`** (e.g. `states=['apo', 'holo']`, with or without `'both'`) to get two independently-numbered runs. Pass `reconcile_apo_holo=True` as well if you also want a `match_states()` reconciliation between them (nearest pocket-centroid matching - apo and holo Global IDs from separate runs aren't otherwise comparable);

`states=['apo', 'holo', 'both']` below runs all three in one call. Drop whichever you don't need. `pdb_ids=PDB_IDS` (defined above) scopes every requested run to a specific, meaningful comparison -- here, one PDB ID per gene, to showcase Global ID conservation/divergence across a diverse set of genes, rather than an unscoped run across every solved structure of the same receptor. `genes=[...]` filters similarly, on top of `pdb_ids`. `all_pockets`/`pocket_summary` themselves are untouched by this -- they're the complete dataset from Step 2.1; only this clustering run is scoped.

In [ ]:
# Re-load from disk here rather than reusing `all_pockets`/`pocket_summary` in memory, so
# this cell also works standalone (e.g. re-running just Step 2.2 later against an existing
# Step 2.1 output).
all_pockets = pocket_io.load_table(SAVING_LOC, 'all_pockets')
pocket_summary = pocket_io.load_table(SAVING_LOC, 'pocket_summary')

GID_ROOT = conf.META_ANALYSIS_ROOT  # global_ID_apo/, global_ID_holo/, global_ID_combined/ each land here

results = gid.run_global_id_states(all_pockets, pocket_summary, states=['apo', 'holo', 'both'],
                                   source_loc=SAVING_LOC, gid_root=GID_ROOT, pdb_ids=PDB_IDS,
                                   formats=FORMATS, make_plots=MAKE_PLOTS,
                                   reconcile_apo_holo=False)  # set True for the apo<->holo Global ID mapping below

pocket_comparison_table = results['both']  # apo+holo clustered together - what most downstream steps want
apo_table, holo_table = results.get('apo'), results.get('holo')  # only present if requested above
pocket_comparison_table.head()

### Apo vs holo pocketome comparison

Already run automatically above, as part of the `'both'` run (`run_apo_holo_comparison()`, requires both states in the same run - reuses `taar_paper_figures/pocketome_metrics.py` for allosteric pocket count/volume/size-class deltas + Jensen-Shannon distance, and `taar_paper_figures/fig2_binding_site.py` for the orthosteric site volume shift). Written to `global_ID_combined/apo_holo_pocketome_summary.csv`.

`results['mapping']` additionally holds the apo<->holo Global ID correspondence (nearest-centroid `match_states()`, written to `apo_holo_global_id_mapping/global_id_state_mapping.csv`) - but only if `states=['apo', 'holo']` were *both* requested above **and** `reconcile_apo_holo=True` was passed. The cell above used `reconcile_apo_holo=False`, so expect `mapping_df` to be `None` below; set it `True` and re-run Step 2.2 to get the mapping.

In [ ]:
apo_holo_summary = results['apo_holo_summary']
apo_holo_summary.head()

#### apo<->holo Global ID mapping
Derived fresh from `results` (not from a variable set in another cell), so this works no matter which cells above were re-run.

In [ ]:
mapping_df = results.get('mapping')
if mapping_df is not None:
    display(mapping_df.head())
else:
    print("mapping_df is None: request states=['apo', 'holo'] (or add 'both') and reconcile_apo_holo=True above to get it")

## End of Step 2
By now `output/meta_analysis/` should hold, e.g.:

```
output/meta_analysis/
├── across_genes/                                  # Step 2.1 output
│   ├── all_pockets.csv, all_pockets.parquet        # every pocket, every frame, every alpha-sphere coordinate
│   ├── pocket_summary.csv, pocket_summary.parquet  # one row per pocket (first frame)
│   ├── orthosteric_perframe_volumes.csv
│   └── plots_largest/, plots_orthosteric/, plots_comparison/
├── global_ID_combined/                             # Step 2.2 'both' run
│   ├── pocket_comparison_table.csv / .parquet       # one row per Local Pocket ID, with Global ID
│   ├── global_pockets_IoU_voxel.csv                 # point-cloud + voxel_group_id, for QC/downstream reuse
│   ├── apo_holo_pocketome_summary.csv
│   ├── unique_to_gene_comparison.csv, unique_to_structure_comparison.csv, shared_in_all_gene_comparison.csv
│   └── pocket_clusters_qc.html, <PDBID>_within_pdb_global_id_plot.html, upsetplot_gene_comparison.png, ...
├── global_ID_apo/, global_ID_holo/                 # Step 2.2 'apo'/'holo' runs, if requested - same layout as above
└── apo_holo_global_id_mapping/                     # only if both 'apo' and 'holo' were requested
    └── global_id_state_mapping.csv                  # nearest-centroid Global ID correspondence between them
```

That's everything the pipeline itself produces. The final section below turns it into the publication figures.

## Step 3 (optional) - the paper figures

`taar_paper_figures/paper_plots.py` is a single, separate entry point that turns the tables written above into the actual publication figures (see `taar_paper_figures/paper_figures_README.md` for the full breakdown of inputs/outputs/panels). It reads `pocket_summary`, `orthosteric_perframe_volumes` (from Step 2.1) and the per-replicate `RMSD_protein_and_name_CA.csv` files (from Step 1), and produces:

- **Figure 1** - RMSD heatmaps, apo vs holo
- **Figure 2** - orthosteric binding site (per-frame volume, Delta, renders)
- **Figure 3** - allosteric pocketome (size/stability/count/class deltas)

written to `output/meta_analysis/paper_figures/`.


In [ ]:
import subprocess

subprocess.run(
    [sys.executable, 'paper_plots.py'],
    cwd=os.path.join(conf.PROJECT_ROOT, 'taar_paper_figures'),
    check=True,
)